# 05 LushProtein Retention Playbook — Two Bets

**A client recommendation: not a menu of tactics, but the two strategies we would stake the budget on.**

LushProtein (Singapore, est. 2013) sells Clear / Lean / Plant / Soy / Better Whey proteins, Collagen Glow, Creatine, Multivitamin and Pure Burn supplements, shakers, a 5-sachet **Discovery Sampler**, and a free-shipping **subscription**.

Notebooks `01`–`04` re-pointed the market basket analysis at one question: *which product combinations and sequences turn a one-and-done buyer into a repeat, subscribed, high-LTV customer?* That work surfaced many possible plays. This notebook does the harder thing — it **picks two**.

We pick two because the data contains exactly **two dominant retention effects**, each an order of magnitude larger than any merchandising tweak. Every other idea (replenishment flows, sampler funnels, win-backs, margin guardrails) is a *mechanic that serves one of these two bets*, not a strategy in its own right. The two bets are:

1. **The Stack Ladder** — move single-product buyers up the *category-breadth* curve.
2. **The Subscription Flywheel** — convert the right routines into *subscriptions*.

The rest of this notebook proves, with live numbers, why these are the best possible bets and how MBA makes them executable.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA = PROJECT_ROOT / "EDA" / "outputs"
MBA = PROJECT_ROOT / "product_mba" / "outputs"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

customers = pd.read_parquet(EDA / "customers.parquet")
breadth = pd.read_csv(EDA / "04_cross_product_ltv.csv")
sub_ltv = pd.read_csv(EDA / "06_sub_vs_onetime_ltv.csv")
anchors = pd.read_csv(MBA / "retention_anchor_products.csv")
scored_handle = pd.read_csv(MBA / "mba_rules_retention_scored_handle.csv")
seq = pd.read_csv(MBA / "mba_sequential_rules.csv")
gateway_flavor = pd.read_csv(MBA / "gateway_product_scorecard.csv")
triggers = pd.read_csv(MBA / "replenishment_triggers.csv")

BASE_REPEAT = customers["is_repeat"].mean()
BASE_SUB = customers["ever_subscribed"].mean()
print(f"Base: {len(customers):,} customers | repeat {BASE_REPEAT:.0%} | subscription {BASE_SUB:.0%}")

Base: 13,780 customers | repeat 32% | subscription 8%


## Why only two? The data has two dominant effects

Before choosing, look at the two strongest, independently-measured relationships in the entire customer base. Everything else is noise next to these.

In [2]:
one  = breadth.set_index("category_label").loc["1 product"]
two  = breadth.set_index("category_label").loc["2 products"]
three= breadth.set_index("category_label").loc["3 products"]
four = breadth.set_index("category_label").loc["4+ products"]
sub  = sub_ltv[sub_ltv["ever_subscribed"] == "Subscriber"].iloc[0]
non  = sub_ltv[sub_ltv["ever_subscribed"].str.contains("One-time")].iloc[0]

print("EFFECT 1 — THE BREADTH CLIFF  (how many distinct categories a customer buys)")
for label, r in [("1 cat", one), ("2 cat", two), ("3 cat", three), ("4+ cat", four)]:
    print(f"   {label:<6} {int(r.customers):>5,} customers | repeat {r.repeat_rate:5.0%} "
          f"| LTV ${r.avg_ltv:6.0f} | {r.avg_orders:.1f} orders")
print(f"   --> repeat rate {one.repeat_rate:.0%} -> {three.repeat_rate:.0%}; "
      f"LTV ${one.avg_ltv:.0f} -> ${three.avg_ltv:.0f} ({three.avg_ltv/one.avg_ltv:.1f}x) "
      f"just by adding categories")

print("\nEFFECT 2 — THE SUBSCRIPTION JACKPOT")
print(f"   Subscriber     {int(sub.customers):>6,} | repeat {sub.repeat_rate:5.0%} "
      f"| LTV ${sub.avg_ltv:6.0f} | {sub.avg_orders:.1f} orders | "
      f"lifespan {sub.avg_lifespan_days:.0f}d")
print(f"   Non-subscriber {int(non.customers):>6,} | repeat {non.repeat_rate:5.0%} "
      f"| LTV ${non.avg_ltv:6.0f} | {non.avg_orders:.1f} orders | "
      f"lifespan {non.avg_lifespan_days:.0f}d")
print(f"   --> a subscriber is worth {sub.avg_ltv/non.avg_ltv:.1f}x and lives "
      f"{sub.avg_lifespan_days/non.avg_lifespan_days:.1f}x longer")

EFFECT 1 — THE BREADTH CLIFF  (how many distinct categories a customer buys)
   1 cat  9,537 customers | repeat   24% | LTV $   170 | 1.5 orders
   2 cat  2,679 customers | repeat   40% | LTV $   230 | 2.2 orders
   3 cat  1,052 customers | repeat   65% | LTV $   470 | 3.4 orders
   4+ cat   512 customers | repeat   89% | LTV $   743 | 7.1 orders
   --> repeat rate 24% -> 65%; LTV $170 -> $470 (2.8x) just by adding categories

EFFECT 2 — THE SUBSCRIPTION JACKPOT
   Subscriber      1,095 | repeat   74% | LTV $   532 | 4.9 orders | lifespan 399d
   Non-subscriber 12,685 | repeat   29% | LTV $   200 | 1.7 orders | lifespan 93d
   --> a subscriber is worth 2.7x and lives 4.3x longer


**The decisive point:** these two effects also describe where the customers actually *are*. The base is overwhelmingly stuck at the bottom of both curves — which is precisely why moving them is the highest-leverage thing LushProtein can do.

In [3]:
n_total = breadth["customers"].sum()
n_1cat = int(one.customers)
n_nonsub = int(non.customers)
print(f"{n_1cat:,} of {n_total:,} customers ({n_1cat/n_total:.0%}) buy only ONE category")
print(f"{n_nonsub:,} of {n_total:,} customers ({n_nonsub/n_total:.0%}) have NEVER subscribed")
print("\n--> The prize is not winning new customers; it is moving the ones we already have")
print("    up these two curves. That is what the two mega-strategies do.")

9,537 of 13,780 customers (69%) buy only ONE category
12,685 of 13,780 customers (92%) have NEVER subscribed

--> The prize is not winning new customers; it is moving the ones we already have
    up these two curves. That is what the two mega-strategies do.


## The size of the prize (why these two beat any merchandising tweak)

Illustrative LTV upside from modest conversion rates, using the *measured* per-segment LTV gaps. These are directional planning numbers, not forecasts.

In [4]:
gap_1to2 = two.avg_ltv - one.avg_ltv        # moving a 1-cat buyer to 2 categories
gap_1to3 = three.avg_ltv - one.avg_ltv      # moving a 1-cat buyer to 3 categories
gap_sub  = sub.avg_ltv - non.avg_ltv        # converting a non-subscriber

print("STACK LADDER — addressable: 9,537 one-category buyers (gap to 2-cat = "
      f"${gap_1to2:.0f}, to 3-cat = ${gap_1to3:.0f})")
for rate in (0.05, 0.10, 0.20):
    movers = n_1cat * rate
    print(f"   move {rate:>3.0%} up one rung -> {movers:,.0f} customers x ${gap_1to2:.0f} "
          f"= +${movers*gap_1to2:,.0f} LTV")

print("\nSUBSCRIPTION FLYWHEEL — addressable: 12,685 non-subscribers (gap = "
      f"${gap_sub:.0f} each)")
for rate in (0.03, 0.05, 0.10):
    conv = n_nonsub * rate
    print(f"   convert {rate:>3.0%} -> {conv:,.0f} subscribers x ${gap_sub:.0f} "
          f"= +${conv*gap_sub:,.0f} LTV")
print("\nBoth prizes run into the hundreds of thousands of SGD in LTV from single-digit "
      "conversion of customers LushProtein ALREADY has.")

STACK LADDER — addressable: 9,537 one-category buyers (gap to 2-cat = $60, to 3-cat = $300)
   move  5% up one rung -> 477 customers x $60 = +$28,480 LTV
   move 10% up one rung -> 954 customers x $60 = +$56,960 LTV
   move 20% up one rung -> 1,907 customers x $60 = +$113,920 LTV

SUBSCRIPTION FLYWHEEL — addressable: 12,685 non-subscribers (gap = $332 each)
   convert  3% -> 381 subscribers x $332 = +$126,362 LTV
   convert  5% -> 634 subscribers x $332 = +$210,603 LTV
   convert 10% -> 1,268 subscribers x $332 = +$421,205 LTV

Both prizes run into the hundreds of thousands of SGD in LTV from single-digit conversion of customers LushProtein ALREADY has.


---
## MEGA-STRATEGY 1 — The Stack Ladder

> **Bet:** systematically walk single-product buyers up the category-breadth curve (1 → 2 → 3), using MBA to choose the *right* next product, at the *right* moment.

### Why it is the best possible breadth bet — by the numbers
- **Biggest addressable segment.** 69% of all customers (9,537) sit on rung 1. No other lever touches this many people.
- **Largest, monotonic marginal effect.** Each rung is a step-change, not a nudge: repeat rate **24% → 40% → 65% → 89%**, LTV **$170 → $230 → $470 → $743**. The curve never flattens — there is always upside to the next rung.
- **MBA makes it causal, not generic.** We don't guess the next product. The **retention-anchor** table ranks items by how much *any* basket containing them lifts downstream repeat/subscription; the **sequential rules** show the real graduation paths (e.g. single-serve → full-size at **2.97× next-order lift**); the **replenishment calendar** says *when* to prompt. Generic "you may also like" modules can't do this.
- **Cheapest to ship.** It reuses rules already computed and maps onto existing Shopify modules + Klaviyo/Recharge flows.

### What rolls up under this one bet
Complete-Your-Stack module · replenishment-timed cross-sell · Discovery-Sampler/gateway front door · win-back that *widens* the basket · margin tie-breaker. They are all instruments for the single goal of advancing a customer one rung.

In [5]:
print("WHAT TO PUSH — retention-anchor products (lift over the "
      f"{BASE_REPEAT:.0%} baseline repeat rate):")
print(anchors[anchors["level"] == "handle"].head(6)[
    ["item", "total_co_orders", "wtd_repeat_uplift",
     "wtd_subscription_uplift", "wtd_ltv_ratio"]].to_string(index=False))

print("\nTHE GRADUATION PATHS — what customers reach for NEXT (sequential, handle level):")
g = seq[(seq["level"] == "handle") & (seq["antecedent"] != seq["consequent"]) &
        (seq["next_order_lift"] >= 1.2)].sort_values("next_order_lift", ascending=False)
print(g.head(8)[["antecedent", "consequent", "co_customers",
                 "next_order_confidence", "next_order_lift"]].to_string(index=False))

print("\nWHEN TO PROMPT — replenishment calendar (fire ~7d before run-out):")
cal = triggers.dropna(subset=["handle"]).sort_values("trigger_day")
for _, r in cal.head(8).iterrows():
    x = r["recommended_cross_sell"]
    tail = f" + introduce {x}" if isinstance(x, str) and x == x else " (reorder reminder)"
    print(f"   Day {int(r['trigger_day']):>3}: reorder {r['handle']}{tail}")

WHAT TO PUSH — retention-anchor products (lift over the 32% baseline repeat rate):
                           item  total_co_orders  wtd_repeat_uplift  wtd_subscription_uplift  wtd_ltv_ratio
                  collagen-glow               51               0.55                     0.18           5.86
                   lean-protein               46               0.45                     0.19           5.97
             multivitamin-vegan              100               0.47                     0.18           3.05
                  super-omega-3              217               0.47                     0.16           4.15
micronized-creatine-monohydrate               66               0.43                     0.12           4.50
     green-tea-extract-capsules              152               0.40                     0.12           5.66

THE GRADUATION PATHS — what customers reach for NEXT (sequential, handle level):
                               antecedent                                conseq

---
## MEGA-STRATEGY 2 — The Subscription Flywheel

> **Bet:** convert the *specific routines that already behave like subscriptions* into subscribe-and-save, defaulting the anchor item to subscription at the second purchase.

### Why it is the best possible value bet — by the numbers
- **Highest value per conversion in the business.** A subscriber is worth **+$332 LTV** (2.7×), repeats at **74% vs 29%**, places **4.9 vs 1.7 orders**, and stays **399 vs 93 days** — a 4.3× longer relationship. Nothing else in the data moves a single customer this much.
- **Enormous untapped headroom.** Only 8% subscribe; 12,685 customers never have. The ceiling is the entire base.
- **MBA tells us *who* will convert — so we don't discount everyone.** The pairs below are **3–4× more subscribed than the 8% baseline** *before* we even ask them. Targeting subscribe-and-save at these routines (vs a blanket offer) protects margin and lifts take-rate.
- **It compounds with Strategy 1.** A subscription is just a *locked-in stack* — so the breadth ladder is the on-ramp and the flywheel is the lock-in. Each makes the other cheaper.

### What rolls up under this one bet
Subscription-anchor offers on high-propensity combos · sampler→subscription funnel · "your routine, monthly, free shipping" bundles · default-to-subscribe on the anchor SKU at order #2.

In [6]:
sub_combos = (scored_handle.sort_values("subscription_uplift", ascending=False)
              .drop_duplicates(subset=["antecedent", "consequent"]).head(8))
print(f"ROUTINES PRIMED TO SUBSCRIBE (baseline subscription = {BASE_SUB:.0%}):")
print(sub_combos[["antecedent", "consequent", "co_orders",
                  "both_item_pct_subscribed", "subscription_uplift",
                  "ltv_ratio"]].to_string(index=False))

print("\nSAMPLER / TRIAL AS THE SUBSCRIPTION ON-RAMP (sequential graduation):")
samp = seq[(seq["antecedent"].str.contains("single|sampler|sachet", case=False, na=False)) &
           (seq["antecedent"] != seq["consequent"])].sort_values("next_order_lift",
                                                                  ascending=False)
print(samp.head(5)[["antecedent", "consequent", "co_customers",
                    "next_order_confidence", "next_order_lift"]].to_string(index=False)
      if len(samp) else "   (none cleared thresholds)")

ROUTINES PRIMED TO SUBSCRIBE (baseline subscription = 8%):
                  antecedent                                consequent  co_orders  both_item_pct_subscribed  subscription_uplift  ltv_ratio
  green-tea-extract-capsules                              lean-protein         46                      0.27                 0.19       5.97
          multivitamin-vegan                             super-omega-3         50                      0.26                 0.18       3.05
               super-omega-3                        multivitamin-vegan         50                      0.26                 0.18       3.05
               super-omega-3                             collagen-glow         51                      0.26                 0.18       5.86
               super-omega-3           micronized-creatine-monohydrate         66                      0.20                 0.12       4.50
  green-tea-extract-capsules              pureburn-fat-burner-capsules         53                    

---
## Why these two — and not the other ideas

| Candidate | Verdict | Reason (data) |
|---|---|---|
| **Stack Ladder** | **Bet** | Largest segment (69% on rung 1) × largest monotonic effect (repeat 24%→89%, LTV up to 4.4×). |
| **Subscription Flywheel** | **Bet** | Largest value per head (+$332 / 2.7× LTV, 4.3× lifespan) × 92% untapped. |
| Replenishment flows | *Mechanic of Bet 1/2* | The *timing* that fires the cross-sell / reorder — not a goal by itself. |
| Sampler funnel | *Mechanic of Bet 1/2* | The *front door* feeding both ladders (single-serve → full-size at 2.97× lift). |
| Win-back | *Mechanic of Bet 1* | Re-entry that should widen the basket toward rung 2. |
| Margin guardrail | *Tie-breaker only* | Margin is directional (31.6% line coverage); a constraint, not a strategy. |

**They reinforce each other.** Breadth builds the multi-product routine; subscription locks that routine in. A 3-category subscriber is the end-state both bets aim at — and the data already shows that customer (4+ cat / subscriber) repeats ~89% and carries 3–4× the LTV of the median. Run them as one funnel: **ladder up, then lock in.**

## Measurement & limitations
- **Measure with holdouts.** Primary KPIs: % of customers reaching 2+/3+ categories (Bet 1) and subscription conversion rate (Bet 2). The segment tables above are the baselines to beat; AOV is explicitly *not* the success metric.
- **Limitations:** only ~52–55% of orders carry analysis-ready item detail, so affinity is under-counted; margin is directional (`00_margin_data.ipynb`); the large `Unknown`/legacy category inflates some first-product stats; sequential rules cover first→second order only; analysis is pooled across SG/HK/MY (per-market splits are the next iteration). The prize figures are illustrative LTV-gap math, not forecasts.

**Bottom line:** the same market basket analysis, re-pointed from "biggest basket" to "stickiest basket", converges on two bets — **ladder up, then lock in** — that together address ~70–90% of the base and run into six figures of LTV from single-digit conversion of customers LushProtein already owns.